In [1]:
# This notebook demonstrates how to use ROSALIA to analyze stray light in a simple exposure. 
import rosalia as rs
import numpy as np
import rosalia as rs
import matplotlib.pyplot as plt# os.system("cp -v /Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-09-16T0933_BAK/MAST/Roman/r0102001002001005001_0004_wfi*.asdf /Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-09-16T0933/MAST/Roman/" )
from matplotlib.colors import SymLogNorm

path = "/Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-09-16T0933/MAST/Roman/r0102001002001005001_0004_wfi*_f158_cal.asdf"
custom_roman_exposure = rs.core.exposure(filename=path) 
# straylight_out = custom_roman_exposure.straylight(verbose=True)


Retrieving exposure : 100%|██████████| 15/15 [00:03<00:00,  3.87it/s]


Currently estimated through pysiaf and quaternion. This is a temporary fix until MAST fixes the RA_TARG, DEC_TARG, PA keywords in the Roman/WFI fits files.


In [2]:
custom_roman_exposure.PIXSCALE

[np.float64(0.10616629949953538),
 np.float64(0.10772625655002653),
 np.float64(0.10597015125601444),
 np.float64(0.10771370521545147),
 np.float64(0.10599281913280478),
 np.float64(0.10766294687884786),
 np.float64(0.10594921410702281),
 np.float64(0.1089867130649018),
 np.float64(0.1093074841021687),
 np.float64(0.1076759288562077),
 np.float64(0.10919729830094209),
 np.float64(0.10916865045944116),
 np.float64(0.10768829890778124),
 np.float64(0.10593980961915085),
 np.float64(0.10931268930920635)]

In [ ]:
print("The filter " + custom_roman_exposure.FILTER + "/" + custom_roman_exposure.TELESCOP +  "/" +  custom_roman_exposure.INSTRUME + "/WFI was not found. Photometric calculations can be compromised.")


In [ ]:
custom_roman_exposure.DETECTOR

In [ ]:
custom_roman_exposure.zodiacal(zody_mode="stsci")

In [ ]:
# Quick aperture photometry 
# dragon_flux_sum = 520495.06 # DN/s	2.23559
# background_sum = 347943.57	# DN/s 	1.59042

In [ ]:
custom_roman_exposure.SCIEXTS

In [ ]:
# Index 3 holds the affected exposure. 
image = custom_roman_exposure.DATA[3]
sky = np.nanmedian(custom_roman_exposure.DATA[3])
skycor = image[0:1500, 500:1500] - sky

fig, ax = plt.subplots(figsize=(8,8))
im = ax.imshow(skycor, origin="lower",
               norm=SymLogNorm(linthresh=1, vmin=-0.2, vmax=20))
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)      
ax.set_title("r0102001002001005001_0004 F158 WFI06", fontsize=10)
plt.tight_layout()
plt.savefig("r0102001002001005001_0004_F158_WFI06.png", dpi=300)


In [ ]:
# Make a radial profile
# First save the image to a fits file
drz_data, drz_wcs, outname = custom_roman_exposure.save_drz(outname="dragon_snap.fits",  
                               data_list=[custom_roman_exposure.DATA[3]-sky], 
                               wcs_list=[custom_roman_exposure.ASTROPYWCS[3]], resolution=0.11)



In [ ]:
ra_dragon_center = 229.6405032
dec_dragon_center = -3.5874811
# radial_mask = rs.utils.create_radial_mask(xsize=image.shape[0], ysize=image.shape[1], q=0.5, theta=-45, center=[2549.2, 4867.4], radius=None)
rbins = np.linspace(0, 2000, 250)

xcen = 2551
ycen = 4856
pa=22
q = 0.25
profile = rs.utils.make_profile(image="dragon_snap.fits", ext=0, xcen=2551, ycen=4856, rbins=rbins,
                                q=q, theta=pa)

rs.utils.save_fits(profile["model"], "radial_mask.fits")



In [ ]:
fig, ax = plt.subplots(figsize=(8,8))


ysnap = 4000
xsnap = 2200
width = 1000
im = ax.imshow(drz_data[ysnap:ysnap+width,xsnap:xsnap+width], origin="lower", norm=SymLogNorm(linthresh=1, vmin=-0.2, vmax=20))
from matplotlib.patches import Ellipse
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=r'Flux (DN/s)')      
ax.set_title("r0102001002001005001_0004 F158 WFI06", fontsize=14)

radius = np.linspace(0,1500,25)
for r in radius:
    ellipse = Ellipse(xy=(xcen-xsnap, ycen-ysnap), width=q*r, height=r, angle=pa, alpha=0.5,
                    edgecolor='white', linestyle='--', facecolor='none', linewidth=1)
    ax.add_patch(ellipse)

ax.set_xlabel("X (pixel)")
ax.set_ylabel("Y (pixel)")
plt.savefig("r0102001002001005001_0004_Dragon_ellipses.png", dpi=300)
plt.show()


In [ ]:
# profile["model"].shape
plt.imshow(drz_data, origin="lower", vmin=0, vmax=1)
plt.show()
plt.imshow(profile["model"], origin="lower", vmin=0, vmax=1)
plt.show()
plt.imshow(drz_data - profile["model"], origin="lower", vmin=0, vmax=1)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
ax.plot(profile["profile"]["r"], profile["profile"]["int"], linewidth=2, color="dodgerblue",
        label="r0102001002001005001_0004 \n F158 WFI06 Dragon's breath")
ax.fill_between(x=profile["profile"]["r"], 
                y1=profile["profile"]["int_s1down"],
                y2=profile["profile"]["int_s1up"], alpha=0.2, color="dodgerblue")

ax.set_ylim(0, 15)
ax.set_xlim(0,500)
ax.set_ylabel("Surface brightness profile (DN/s)")
ax.set_xlabel("Radius (pixel)")
ax.legend()
plt.tight_layout()
plt.savefig("r0102001002001005001_0004_Dragon_profile.png", dpi=300)